# Algoritmos Genéticos: Asesor de Inversiones
**Autor:** Lizbeth Hernández

---

## Descripción del Problema

Una empresa ofrece a sus clientes invertir en bonos de varias compañías. Cada bono tiene:
- **Precio por bono**
- **Rendimiento del bono**
- **Cantidad de bonos disponibles**

El objetivo es encontrar la combinación de bonos que **maximice el rendimiento total**, sin exceder el capital disponible del cliente.

### Tabla 37. Tipos de bonos disponibles

| Bono | Precio | Cantidad | Rendimiento |
|------|--------|----------|-------------|
| A    | 500    | 5        | 0.15        |
| B    | 700    | 6        | 0.05        |
| C    | 300    | 10       | 0.03        |
| D    | 800    | 3        | 0.10        |
| E    | 400    | 7        | 0.10        |

> **Referencia:** Benítez Iglésias, R. (2014). *Inteligencia artificial avanzada*. Editorial UOC.

---
## Parte 1: Algoritmo Genético desde cero (implementación manual)

Implementación basada en la bibliografía de Benítez Iglésias (2014).

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────
# Datos del problema
# ─────────────────────────────────────────
bonos = {
    'nombre':       ['A',    'B',    'C',    'D',    'E'],
    'precio':       [500,    700,    300,    800,    400],
    'cantidad_max': [5,      6,      10,     3,      7],
    'rendimiento':  [0.15,   0.05,   0.03,   0.10,   0.10]
}

CAPITAL = 10000   # Capital disponible del cliente
N_BONOS = len(bonos['nombre'])

print("=" * 50)
print("DATOS DEL PROBLEMA")
print("=" * 50)
print(f"Capital disponible: ${CAPITAL:,}")
print(f"Número de tipos de bonos: {N_BONOS}")
print()
print(f"{'Bono':<6} {'Precio':>8} {'Cantidad':>10} {'Rendimiento':>13}")
print("-" * 40)
for i in range(N_BONOS):
    print(f"{bonos['nombre'][i]:<6} {bonos['precio'][i]:>8} {bonos['cantidad_max'][i]:>10} {bonos['rendimiento'][i]:>13.2%}")

### 1.1 Representación del cromosoma

Cada individuo (cromosoma) es un vector de 5 genes, donde cada gen representa la **cantidad de bonos** a comprar de cada tipo:

```
cromosoma = [a, b, c, d, e]
           donde a ∈ [0..5], b ∈ [0..6], c ∈ [0..10], d ∈ [0..3], e ∈ [0..7]
```

In [ ]:
# ─────────────────────────────────────────
# FUNCIÓN OBJETIVO
# Calcula el rendimiento total de una cartera
# ─────────────────────────────────────────
def funcion_objetivo(cromosoma):
    """
    Calcula el rendimiento total de una combinación de bonos.
    Rendimiento_i = precio_i * cantidad_i * rendimiento_i
    """
    rendimiento_total = 0
    for i in range(N_BONOS):
        rendimiento_total += bonos['precio'][i] * cromosoma[i] * bonos['rendimiento'][i]
    return rendimiento_total


# ─────────────────────────────────────────
# FITNESS
# Penaliza soluciones que exceden el capital
# ─────────────────────────────────────────
def fitness(cromosoma):
    """
    Función de aptitud: retorna el rendimiento si la solución
    es factible (costo <= CAPITAL), de lo contrario retorna 0.
    """
    costo_total = sum(bonos['precio'][i] * cromosoma[i] for i in range(N_BONOS))
    if costo_total > CAPITAL:
        return 0  # Solución no factible → penalización total
    return funcion_objetivo(cromosoma)


def costo_cartera(cromosoma):
    return sum(bonos['precio'][i] * cromosoma[i] for i in range(N_BONOS))


# Prueba rápida
ejemplo = [2, 1, 3, 1, 2]
print(f"Cromosoma de ejemplo: {ejemplo}")
print(f"  Costo total:       ${costo_cartera(ejemplo):,}")
print(f"  Rendimiento:       ${funcion_objetivo(ejemplo):,.2f}")
print(f"  Fitness:           ${fitness(ejemplo):,.2f}")

### 1.2 Operadores Genéticos (implementación manual)

In [ ]:
# ─────────────────────────────────────────
# INICIALIZACIÓN DE POBLACIÓN
# ─────────────────────────────────────────
def crear_individuo():
    """Crea un individuo aleatorio respetando los límites de cada bono."""
    return [random.randint(0, bonos['cantidad_max'][i]) for i in range(N_BONOS)]


def crear_poblacion(tam_poblacion):
    return [crear_individuo() for _ in range(tam_poblacion)]


# ─────────────────────────────────────────
# SELECCIÓN: Torneo
# ─────────────────────────────────────────
def seleccion_torneo(poblacion, k=3):
    """
    Selección por torneo: se escogen k individuos al azar
    y se retorna el mejor (mayor fitness).
    """
    candidatos = random.sample(poblacion, k)
    return max(candidatos, key=fitness)


# ─────────────────────────────────────────
# CRUZA: Un punto de corte
# ─────────────────────────────────────────
def cruza_un_punto(padre1, padre2):
    """
    Cruza de un punto: se elige un punto aleatorio
    y se intercambian los segmentos entre padres.
    """
    punto = random.randint(1, N_BONOS - 1)
    hijo1 = padre1[:punto] + padre2[punto:]
    hijo2 = padre2[:punto] + padre1[punto:]
    return hijo1, hijo2


# ─────────────────────────────────────────
# MUTACIÓN: Gen aleatorio
# ─────────────────────────────────────────
def mutacion(individuo, prob_mutacion=0.1):
    """
    Mutación: con probabilidad prob_mutacion, se reemplaza
    el valor de un gen por un número aleatorio válido.
    """
    individuo = individuo[:]
    for i in range(N_BONOS):
        if random.random() < prob_mutacion:
            individuo[i] = random.randint(0, bonos['cantidad_max'][i])
    return individuo


print("Operadores genéticos definidos correctamente.")
p1 = crear_individuo()
p2 = crear_individuo()
h1, h2 = cruza_un_punto(p1, p2)
print(f"  Padre 1: {p1}")
print(f"  Padre 2: {p2}")
print(f"  Hijo 1:  {h1}")
print(f"  Hijo 2:  {h2}")
print(f"  Mutado:  {mutacion(h1)}")

### 1.3 Ciclo del Algoritmo Genético

In [ ]:
# ─────────────────────────────────────────
# ALGORITMO GENÉTICO PRINCIPAL
# ─────────────────────────────────────────
def algoritmo_genetico(tam_poblacion=50, n_generaciones=100,
                        prob_cruza=0.8, prob_mutacion=0.1, semilla=42):
    random.seed(semilla)
    
    # Inicialización
    poblacion = crear_poblacion(tam_poblacion)
    
    historial_mejor = []
    historial_promedio = []
    mejor_global = None
    mejor_fitness_global = 0

    for gen in range(n_generaciones):
        # Evaluar fitness de la población
        fitnesses = [fitness(ind) for ind in poblacion]
        
        # Registrar estadísticas
        mejor_gen = max(fitnesses)
        promedio_gen = np.mean(fitnesses)
        historial_mejor.append(mejor_gen)
        historial_promedio.append(promedio_gen)
        
        # Actualizar mejor global
        idx_mejor = fitnesses.index(mejor_gen)
        if mejor_gen > mejor_fitness_global:
            mejor_fitness_global = mejor_gen
            mejor_global = poblacion[idx_mejor][:]
        
        # Crear nueva generación
        nueva_poblacion = []
        
        # Elitismo: conservar el mejor individuo
        nueva_poblacion.append(poblacion[idx_mejor][:])
        
        while len(nueva_poblacion) < tam_poblacion:
            # Selección
            padre1 = seleccion_torneo(poblacion)
            padre2 = seleccion_torneo(poblacion)
            
            # Cruza
            if random.random() < prob_cruza:
                hijo1, hijo2 = cruza_un_punto(padre1, padre2)
            else:
                hijo1, hijo2 = padre1[:], padre2[:]
            
            # Mutación
            hijo1 = mutacion(hijo1, prob_mutacion)
            hijo2 = mutacion(hijo2, prob_mutacion)
            
            nueva_poblacion.extend([hijo1, hijo2])
        
        poblacion = nueva_poblacion[:tam_poblacion]
    
    return mejor_global, mejor_fitness_global, historial_mejor, historial_promedio


# Ejecutar el AG
mejor_manual, fitness_manual, hist_mejor, hist_prom = algoritmo_genetico(
    tam_poblacion=60, n_generaciones=150, prob_cruza=0.85, prob_mutacion=0.12
)

print("=" * 50)
print("RESULTADO - Algoritmo Genético Manual")
print("=" * 50)
print(f"Mejor cromosoma: {mejor_manual}")
print()
print(f"{'Bono':<6} {'Cantidad':>10} {'Precio':>8} {'Inversión':>12} {'Rendimiento $':>15}")
print("-" * 55)
for i in range(N_BONOS):
    inversion = bonos['precio'][i] * mejor_manual[i]
    rend = inversion * bonos['rendimiento'][i]
    print(f"{bonos['nombre'][i]:<6} {mejor_manual[i]:>10} {bonos['precio'][i]:>8} ${inversion:>11,} ${rend:>14,.2f}")
print("-" * 55)
print(f"{'TOTAL':<6} {'':>10} {'':>8} ${costo_cartera(mejor_manual):>11,} ${fitness_manual:>14,.2f}")
print(f"\nCapital utilizado: ${costo_cartera(mejor_manual):,} / ${CAPITAL:,}")
print(f"Capital restante:  ${CAPITAL - costo_cartera(mejor_manual):,}")

### 1.4 Visualización de Resultados (Implementación Manual)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Algoritmo Genético Manual - Asesor de Inversiones', fontsize=14, fontweight='bold')

# ── Gráfica 1: Convergencia del AG
ax1 = axes[0]
ax1.plot(hist_mejor, label='Mejor fitness', color='#1565C0', linewidth=2)
ax1.plot(hist_prom, label='Fitness promedio', color='#42A5F5', linewidth=1.5, linestyle='--')
ax1.set_title('Convergencia del Algoritmo Genético')
ax1.set_xlabel('Generación')
ax1.set_ylabel('Fitness (Rendimiento $)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Gráfica 2: Cantidad de bonos a comprar
ax2 = axes[1]
colores = ['#1565C0', '#1976D2', '#1E88E5', '#2196F3', '#42A5F5']
barras = ax2.bar(bonos['nombre'], mejor_manual, color=colores, edgecolor='white', linewidth=1.5)
ax2.set_title('Cantidad de Bonos a Comprar (Solución Óptima)')
ax2.set_xlabel('Tipo de Bono')
ax2.set_ylabel('Cantidad')
ax2.set_ylim(0, max(bonos['cantidad_max']) + 1)

# Añadir etiquetas y límites máximos
for i, (barra, cant, maximo) in enumerate(zip(barras, mejor_manual, bonos['cantidad_max'])):
    ax2.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.1,
             str(cant), ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax2.plot([i - 0.4, i + 0.4], [maximo, maximo], 'r--', alpha=0.6, linewidth=1)

ax2.grid(True, axis='y', alpha=0.3)
limite_patch = plt.Line2D([0], [0], color='red', linestyle='--', alpha=0.6, label='Máx disponible')
ax2.legend(handles=[limite_patch])

# ── Gráfica 3: Desglose de inversión y rendimiento
ax3 = axes[2]
inversiones = [bonos['precio'][i] * mejor_manual[i] for i in range(N_BONOS)]
rendimientos = [inversiones[i] * bonos['rendimiento'][i] for i in range(N_BONOS)]

x = np.arange(N_BONOS)
ancho = 0.35
b1 = ax3.bar(x - ancho/2, inversiones, ancho, label='Inversión ($)', color='#1565C0', alpha=0.85)
ax3_twin = ax3.twinx()
b2 = ax3_twin.bar(x + ancho/2, rendimientos, ancho, label='Rendimiento ($)', color='#4CAF50', alpha=0.85)

ax3.set_title('Inversión vs Rendimiento por Bono')
ax3.set_xlabel('Tipo de Bono')
ax3.set_ylabel('Inversión ($)', color='#1565C0')
ax3_twin.set_ylabel('Rendimiento ($)', color='#4CAF50')
ax3.set_xticks(x)
ax3.set_xticklabels(bonos['nombre'])
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax3_twin.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
lines = [b1, b2]
ax3.legend(lines, [l.get_label() for l in lines], loc='upper right')
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/resultado_manual.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Rendimiento total óptimo: ${fitness_manual:,.2f}")

---
## Parte 2: Algoritmo Genético con librería DEAP

DEAP (Distributed Evolutionary Algorithms in Python) es una librería flexible para algoritmos evolutivos.

In [ ]:
from deap import base, creator, tools, algorithms
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────
# CONFIGURACIÓN DE DEAP
# ─────────────────────────────────────────

# Limpiar definiciones previas si el notebook se re-ejecuta
if hasattr(creator, 'FitnessMax'):
    del creator.FitnessMax
if hasattr(creator, 'Individual'):
    del creator.Individual

# Definir el tipo de fitness (maximización)
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Definir el individuo como lista con el fitness anterior
creator.create("Individual", list, fitness=creator.FitnessMax)

# Caja de herramientas
toolbox = base.Toolbox()

# ─────────────────────────────────────────
# FUNCIÓN OBJETIVO DEAP
# Debe retornar una tupla
# ─────────────────────────────────────────
def evaluar_deap(individuo):
    """
    Función de evaluación para DEAP.
    Retorna una tupla (fitness,) — requerido por DEAP.
    Penaliza con 0 si se excede el capital.
    """
    costo = sum(bonos['precio'][i] * individuo[i] for i in range(N_BONOS))
    if costo > CAPITAL:
        return (0,)
    rendimiento = sum(bonos['precio'][i] * individuo[i] * bonos['rendimiento'][i]
                      for i in range(N_BONOS))
    return (rendimiento,)


# Generadores de genes (uno por tipo de bono)
def gen_bono(idx):
    return random.randint(0, bonos['cantidad_max'][idx])

toolbox.register("attr_A", gen_bono, 0)
toolbox.register("attr_B", gen_bono, 1)
toolbox.register("attr_C", gen_bono, 2)
toolbox.register("attr_D", gen_bono, 3)
toolbox.register("attr_E", gen_bono, 4)

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_A, toolbox.attr_B, toolbox.attr_C,
                  toolbox.attr_D, toolbox.attr_E), n=1)

toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# ─────────────────────────────────────────
# OPERADORES DEAP
# ─────────────────────────────────────────
# Selección por torneo (k=3)
toolbox.register("select", tools.selTournament, tournsize=3)

# Cruza de un punto
toolbox.register("mate", tools.cxOnePoint)

# Mutación uniforme con límites por gen
toolbox.register("mutate", tools.mutUniformInt,
                 low=[0]*N_BONOS,
                 up=bonos['cantidad_max'],
                 indpb=0.2)

# Función de evaluación
toolbox.register("evaluate", evaluar_deap)

print("Configuración DEAP completada.")
print(f"  Selección:  Torneo (k=3)")
print(f"  Cruza:      Un punto (cxOnePoint)")
print(f"  Mutación:   Uniforme entera (mutUniformInt, indpb=0.2)")
print(f"  Evaluación: Rendimiento total con penalización por capital excedido")

In [ ]:
# ─────────────────────────────────────────
# EJECUTAR AG CON DEAP
# ─────────────────────────────────────────
random.seed(42)
np.random.seed(42)

TAM_POB = 60
N_GEN   = 150
CXPB    = 0.85   # probabilidad de cruza
MUTPB   = 0.20   # probabilidad de mutación

poblacion_deap = toolbox.population(n=TAM_POB)

# Estadísticas a registrar
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("max",  np.max)
stats.register("mean", np.mean)
stats.register("min",  np.min)

# Hall of Fame: guarda el mejor individuo encontrado
hof = tools.HallOfFame(1)

# Ejecutar el AG con algoritmo simple de DEAP
poblacion_final, logbook = algorithms.eaSimple(
    poblacion_deap, toolbox,
    cxpb=CXPB, mutpb=MUTPB,
    ngen=N_GEN,
    stats=stats, halloffame=hof,
    verbose=False
)

mejor_deap = list(hof[0])
fitness_deap = hof[0].fitness.values[0]

gen_log    = logbook.select("gen")
max_log    = logbook.select("max")
mean_log   = logbook.select("mean")

print("=" * 50)
print("RESULTADO - Algoritmo Genético con DEAP")
print("=" * 50)
print(f"Mejor cromosoma: {mejor_deap}")
print()
print(f"{'Bono':<6} {'Cantidad':>10} {'Precio':>8} {'Inversión':>12} {'Rendimiento $':>15}")
print("-" * 55)
for i in range(N_BONOS):
    inversion = bonos['precio'][i] * mejor_deap[i]
    rend = inversion * bonos['rendimiento'][i]
    print(f"{bonos['nombre'][i]:<6} {mejor_deap[i]:>10} {bonos['precio'][i]:>8} ${inversion:>11,} ${rend:>14,.2f}")
print("-" * 55)
costo_deap = sum(bonos['precio'][i] * mejor_deap[i] for i in range(N_BONOS))
print(f"{'TOTAL':<6} {'':>10} {'':>8} ${costo_deap:>11,} ${fitness_deap:>14,.2f}")
print(f"\nCapital utilizado: ${costo_deap:,} / ${CAPITAL:,}")
print(f"Capital restante:  ${CAPITAL - costo_deap:,}")

### 2.1 Visualización de Resultados (DEAP)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Algoritmo Genético DEAP - Asesor de Inversiones', fontsize=14, fontweight='bold')

# ── Gráfica 1: Convergencia
ax1 = axes[0]
ax1.plot(gen_log, max_log, label='Mejor fitness', color='#2E7D32', linewidth=2)
ax1.plot(gen_log, mean_log, label='Fitness promedio', color='#81C784', linewidth=1.5, linestyle='--')
ax1.set_title('Convergencia (DEAP)')
ax1.set_xlabel('Generación')
ax1.set_ylabel('Fitness (Rendimiento $)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Gráfica 2: Cantidad de bonos
ax2 = axes[1]
colores_g = ['#2E7D32', '#388E3C', '#43A047', '#4CAF50', '#66BB6A']
barras2 = ax2.bar(bonos['nombre'], mejor_deap, color=colores_g, edgecolor='white', linewidth=1.5)
ax2.set_title('Cantidad de Bonos a Comprar (DEAP)')
ax2.set_xlabel('Tipo de Bono')
ax2.set_ylabel('Cantidad')
ax2.set_ylim(0, max(bonos['cantidad_max']) + 1)

for i, (barra, cant, maximo) in enumerate(zip(barras2, mejor_deap, bonos['cantidad_max'])):
    ax2.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.1,
             str(cant), ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax2.plot([i - 0.4, i + 0.4], [maximo, maximo], 'r--', alpha=0.6, linewidth=1)

ax2.grid(True, axis='y', alpha=0.3)
limite_patch = plt.Line2D([0], [0], color='red', linestyle='--', alpha=0.6, label='Máx disponible')
ax2.legend(handles=[limite_patch])

# ── Gráfica 3: Inversión vs Rendimiento
ax3 = axes[2]
inversiones_d = [bonos['precio'][i] * mejor_deap[i] for i in range(N_BONOS)]
rendimientos_d = [inversiones_d[i] * bonos['rendimiento'][i] for i in range(N_BONOS)]

x = np.arange(N_BONOS)
ancho = 0.35
b1d = ax3.bar(x - ancho/2, inversiones_d, ancho, label='Inversión ($)', color='#2E7D32', alpha=0.85)
ax3_twin = ax3.twinx()
b2d = ax3_twin.bar(x + ancho/2, rendimientos_d, ancho, label='Rendimiento ($)', color='#FF8F00', alpha=0.85)

ax3.set_title('Inversión vs Rendimiento por Bono (DEAP)')
ax3.set_xlabel('Tipo de Bono')
ax3.set_ylabel('Inversión ($)', color='#2E7D32')
ax3_twin.set_ylabel('Rendimiento ($)', color='#FF8F00')
ax3.set_xticks(x)
ax3.set_xticklabels(bonos['nombre'])
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax3_twin.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
lines_d = [b1d, b2d]
ax3.legend(lines_d, [l.get_label() for l in lines_d], loc='upper right')
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/resultado_deap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Parte 3: Comparación de Ambas Implementaciones

In [ ]:
print("=" * 70)
print("COMPARACIÓN: Implementación Manual vs DEAP")
print("=" * 70)

tabla = [
    ("Aspecto", "Manual", "DEAP"),
    ("-" * 25, "-" * 20, "-" * 20),
    ("Función objetivo",
     "Suma explícita de precio*cant*rend",
     "Misma lógica, retorna tupla"),
    ("Fitness",
     "Retorna 0 si costo > CAPITAL",
     "Retorna (0,) si costo > CAPITAL"),
    ("Selección",
     "Torneo manual (k=3)",
     "tools.selTournament (k=3)"),
    ("Cruza",
     "Un punto manual",
     "tools.cxOnePoint"),
    ("Mutación",
     "Gen aleatorio por prob",
     "tools.mutUniformInt (indpb=0.2)"),
    ("Elitismo",
     "Manual (conserva mejor)",
     "HallOfFame(1)"),
    ("Ciclo evolutivo",
     "Loop explícito",
     "algorithms.eaSimple"),
    ("Estadísticas",
     "Lista manual",
     "tools.Statistics + Logbook"),
]

for row in tabla:
    print(f"{row[0]:<26} {row[1]:<30} {row[2]:<25}")

print()
print("=" * 70)
print("RESULTADOS NUMÉRICOS")
print("=" * 70)
print(f"{'':30} {'Manual':>15} {'DEAP':>15}")
print("-" * 62)
for i, nombre in enumerate(bonos['nombre']):
    print(f"  Bonos {nombre} comprados:{'':16} {mejor_manual[i]:>15} {mejor_deap[i]:>15}")
print("-" * 62)
print(f"  Costo total:{'':22} ${costo_cartera(mejor_manual):>14,} ${costo_deap:>14,}")
print(f"  Rendimiento total:{'':16} ${fitness_manual:>14,.2f} ${fitness_deap:>14,.2f}")
print(f"  Capital restante:{'':17} ${CAPITAL - costo_cartera(mejor_manual):>14,} ${CAPITAL - costo_deap:>14,}")

In [ ]:
# ─────────────────────────────────────────
# Gráfica comparativa final
# ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparación: Implementación Manual vs DEAP', fontsize=14, fontweight='bold')

# ── Gráfica 1: Convergencia comparativa
ax1 = axes[0]
ax1.plot(range(len(hist_mejor)), hist_mejor, label='Manual', color='#1565C0', linewidth=2)
ax1.plot(gen_log, max_log, label='DEAP', color='#2E7D32', linewidth=2, linestyle='--')
ax1.set_title('Convergencia: Manual vs DEAP')
ax1.set_xlabel('Generación')
ax1.set_ylabel('Mejor Fitness (Rendimiento $)')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── Gráfica 2: Comparación de bonos por solución
ax2 = axes[1]
x = np.arange(N_BONOS)
ancho = 0.35
b1 = ax2.bar(x - ancho/2, mejor_manual, ancho, label='Manual', color='#1565C0', alpha=0.85)
b2 = ax2.bar(x + ancho/2, mejor_deap,   ancho, label='DEAP',   color='#2E7D32', alpha=0.85)

ax2.set_title('Cantidad de Bonos: Manual vs DEAP')
ax2.set_xlabel('Tipo de Bono')
ax2.set_ylabel('Cantidad')
ax2.set_xticks(x)
ax2.set_xticklabels(bonos['nombre'])
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)

for bar in b1:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=10, color='#1565C0')
for bar in b2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(int(bar.get_height())), ha='center', va='bottom', fontsize=10, color='#2E7D32')

# Anotación de rendimiento final
ax2.text(0.98, 0.95, f'Manual: ${fitness_manual:,.2f}\nDEAP:   ${fitness_deap:,.2f}',
         transform=ax2.transAxes, ha='right', va='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
         fontsize=10)

plt.tight_layout()
plt.savefig('/home/claude/comparacion_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Análisis completado.")

---
## Conclusiones

### Diferencias en la función objetivo
- **Manual:** La función `funcion_objetivo()` calcula directamente `Σ precio_i × cantidad_i × rendimiento_i`. Es independiente de la restricción de capital.
- **DEAP:** La función `evaluar_deap()` integra tanto la función objetivo como la penalización, y **debe retornar una tupla** (requerimiento del framework).

### Diferencias en el fitness
- **Manual:** El fitness es una función separada que llama a la función objetivo y aplica penalización (retorna `0` si se excede el capital).
- **DEAP:** El fitness está integrado en la función de evaluación y se almacena en el atributo `ind.fitness.values` de cada individuo. DEAP maneja la comparación de fitness internamente.

### Diferencias en los operadores
| Operador  | Manual                                  | DEAP                                      |
|-----------|-----------------------------------------|-------------------------------------------|
| Selección | Torneo codificado manualmente           | `tools.selTournament` (optimizado)        |
| Cruza     | Intercambio de segmentos codificado     | `tools.cxOnePoint` (librería)             |
| Mutación  | Por cada gen con probabilidad uniforme  | `tools.mutUniformInt` con `indpb` por gen |

### Ventajas de DEAP
- Código más conciso y legible
- Operadores optimizados y probados
- Registro automático de estadísticas con `Logbook`
- `HallOfFame` para preservar las mejores soluciones
- Fácil extensión a algoritmos más avanzados (NSGA-II, CMA-ES, etc.)

### Ventajas de la implementación manual
- Control total sobre cada paso del algoritmo
- Sin dependencias externas
- Más fácil de depurar y entender para fines didácticos